# 03 - Analysis (R)

## 0. Setup

In [ ]:
# install.packages(c("lme4", "emmeans", "performance", "rempsyc", "flextable"))

In [ ]:
library(lme4); library(emmeans); library(performance)
library(rempsyc); library(flextable)

options(scipen = 999, digits = 4)
emm_options(lmer.df = "asymptotic")   # Wald z for every emmeans result (the paper's choice)

In [ ]:
sessionInfo()      # R version and every loaded package

## 1. Helper functions

### Data preparation and model

In [ ]:
prep_data <- function(file) {
  # Read one outlet-week-issue file and code it exactly as the Python model
  d <- read.csv(file.path("../model_dfs", file))
  d <- subset(d, !is.na(simpson_Dz))

  # Explicit level order: never rely on alphabetical sorting, which depends on the locale
  d$country <- factor(d$country, levels = c("BE", "IR", "NL", "TR", "UK"))
  d$camp    <- factor(d$campaign_period, levels = c(0, 1), labels = c("routine", "campaign"))
  d$own     <- factor(ifelse(d$type == "Public broadcaster", "PSM", "newspaper"),
                      levels = c("newspaper", "PSM"))
  d$iss     <- factor(d$topic_label, levels = c("immigration", "rising prices"))
  d$outlet  <- paste(d$country, d$brand, sep = "_")
  d$cweek   <- paste(d$country, d$year_week, sep = "_")

  # Two-level factors -0.5/+0.5 (coefficient = second level minus first), country
  # sum-to-zero (deviation from the five-country average). Named columns make the
  # summary readable: "ownPSM" instead of "own1".
  contrasts(d$camp) <- matrix(c(-0.5, 0.5), dimnames = list(NULL, "campaign"))
  contrasts(d$own)  <- matrix(c(-0.5, 0.5), dimnames = list(NULL, "PSM"))
  contrasts(d$iss)  <- matrix(c(-0.5, 0.5), dimnames = list(NULL, "prices"))
  contrasts(d$country) <- contr.sum(5)
  colnames(contrasts(d$country)) <- c("BE", "IR", "NL", "TR")
  d
}

MODEL_FORMULA <- simpson_Dz ~ country * (camp + own + iss) + (1 | outlet) + (1 | cweek)

### Table output

In [ ]:
TABLE_DIR <- "../results/tables"

save_apa <- function(df, name, number, title, note = "", landscape = FALSE, ...) {
  # landscape = TRUE turns the Word page sideways for wide tables.
  # "..." passes any extra formatting options straight on to nice_table()
  # Same output layout as the Python apa_table: a CSV and a Word file per table.
  for (sub in c("csv", "docx")) dir.create(file.path(TABLE_DIR, sub), recursive = TRUE,
                                           showWarnings = FALSE)
  write.csv(df, file.path(TABLE_DIR, "csv", paste0(name, ".csv")), row.names = FALSE)

  # Display headers; nice_table formats "p" itself and merges CI_lower/CI_upper into
  # "95% CI". The Holm column is formatted here with rempsyc's own p formatter (three
  # decimals, no leading zero); Holm values capped at 1 are shown as "> .999".
  names(df) <- sub("^contrast$", "Contrast", sub("^p_Holm$", "p Holm", names(df)))
  if ("p Holm" %in% names(df))
    df[["p Holm"]] <- ifelse(df[["p Holm"]] > .999, "> .999", format_p(df[["p Holm"]]))
  # With spanning headers ("Issue.Column"), give undotted columns an empty spanner so
  # their labels sit on the same row as the other column labels
  if (isTRUE(list(...)$separate.header))
    names(df) <- ifelse(grepl(".", names(df), fixed = TRUE), names(df), paste0(" .", names(df)))
  ft <- nice_table(df, title = c(paste("Table", number), title), note = note, stars = FALSE, ...)
  ft <- bg(ft, bg = "white", part = "all")     # black text stays readable on a dark theme
  if (landscape) {
    # Compact layout so a 20-row table fits one landscape page: single-spaced rows,
    # fixed column widths in inches: id columns wider; value columns 0.8 when there are
    # many of them, wider when there are few (so headers do not break mid-word)
    ft <- padding(ft, padding.top = 1, padding.bottom = 1, part = "all")
    ft <- line_spacing(ft, space = 1, part = "all")
    ft <- height_all(ft, height = 0.2, part = "all")
    ft <- hrule(ft, rule = "atleast", part = "all")
    ft <- width(ft, j = 1, width = 0.7)
    ft <- width(ft, j = 2, width = 1.8)
    ft <- width(ft, j = 3:ncol(df), width = if (ncol(df) > 6) 0.8 else 1.3)
    ft <- set_table_properties(ft, layout = "fixed")
  }
  page <- officer::prop_section(
    page_size = officer::page_size(orient = if (landscape) "landscape" else "portrait"))
  save_as_docx(ft, path = file.path(TABLE_DIR, "docx", paste0(name, ".docx")), pr_section = page)

  IRdisplay::display_html(as.character(htmltools_value(ft)))   # render in the notebook
  invisible(ft)
}

tidy_emm <- function(x, adjust = "none") {
  # Any emmeans result -> plain data frame with CI and test, ready for nice_table
  s <- as.data.frame(summary(x, infer = TRUE, adjust = adjust))
  names(s) <- sub("^estimate$|^emmean$", "b", names(s))
  names(s) <- sub("asymp.LCL", "CI_lower", sub("asymp.UCL", "CI_upper", names(s)))
  names(s) <- sub("z.ratio", "z", sub("p.value", "p", names(s)))
  s$df <- NULL                          # always Inf under Wald z; carries no information
  s
}

### Model tables

In [ ]:
COUNTRY_NAME <- c(BE = "Belgium", IR = "Ireland", NL = "Netherlands",
                  TR = "Turkey", UK = "United Kingdom")
FACTORS      <- c(camp = "Campaign", own = "Public broadcaster",
                  iss = "Issue (rising prices)")
EFFECT_LABEL <- c(camp = "Campaign vs routine", own = "Public broadcaster vs newspaper",
                  iss = "Rising prices vs immigration")
COLS <- c("b", "SE", "z", "p", "CI_lower", "CI_upper")

model_table <- function(m) {
  # Every fixed effect, with all five countries in each block (UK included).
  # "eff" = each country minus the five-country average; for the interaction blocks,
  # each country's effect minus the average effect, i.e. the country x factor term.
  cn <- levels(m@frame$country)                  # country codes, as in the Python table
  block <- function(x, labels) cbind(Parameter = labels, tidy_emm(x)[, COLS])
  rows <- list(
    block(emmeans(m, ~ 1), "Intercept"),
    block(contrast(emmeans(m, ~ country), "eff"), paste0("Country (", cn, ")")))
  for (f in names(FACTORS)) {
    rows[[length(rows) + 1]] <- block(
      contrast(emmeans(m, as.formula(paste("~", f))), "revpairwise"), FACTORS[[f]])
    rows[[length(rows) + 1]] <- block(
      contrast(emmeans(m, as.formula(paste("~", f, "* country"))),
               interaction = c("revpairwise", "eff")),
      paste0(FACTORS[[f]], " \u00d7 Country (", cn, ")"))
  }
  do.call(rbind, rows)
}

pairs_table <- function(m) {
  # All ten pairwise country differences, unadjusted and Holm-adjusted p side by side
  out <- tidy_emm(pairs(emmeans(m, ~ country)))
  out$p_Holm <- p.adjust(out$p, method = "holm")
  out
}

simple_effects_table <- function(m) {
  # Each effect averaged over the five countries, then within each country
  # (second level minus first: campaign - routine, PSM - newspaper, prices - immigration)
  rows <- lapply(names(EFFECT_LABEL), function(f) {
    avg <- tidy_emm(contrast(emmeans(m, as.formula(paste("~", f))), "revpairwise"))
    by  <- tidy_emm(contrast(emmeans(m, as.formula(paste("~", f, "| country"))), "revpairwise"))
    data.frame(Effect  = c(EFFECT_LABEL[[f]], rep("", nrow(by))),
               Country = c("Five-country average", COUNTRY_NAME[as.character(by$country)]),
               rbind(avg[, COLS], by[, COLS]), row.names = NULL)
  })
  do.call(rbind, rows)
}

### Descriptive tables

In [ ]:
ISSUE_LABEL <- c(immigration = "Immigration", rising_prices = "Rising prices")
PERIOD_LABEL <- c("0" = "Routine", "1" = "Campaign")

TYPE_ORDER  <- c("Quality newspaper", "Popular newspaper", "Public broadcaster")
SOC_CATS    <- c(nr_politicians = "Politics", nr_experts_professionals = "Experts",
                 nr_civil_society = "Civil society", nr_general_public = "Citizens")

read_articles <- function(file) {
  # Article-level actor counts. Rows without article metadata (no outlet or date) are
  # dropped: they cannot be assigned to an outlet-week and are also absent from the models.
  a <- read.csv(file.path("../analysis_data", file), sep = ";")
  a <- subset(a, !is.na(brand) & brand != "")
  a$country <- factor(a$country, levels = names(COUNTRY_NAME))
  a$type    <- factor(a$type, levels = TYPE_ORDER)
  a$outlet  <- paste(a$country, a$brand, sep = "_")
  a
}

outlet_frame <- function(a) {
  # One row per outlet, ordered by country, then outlet type, then name (as in Table C1).
  # Country codes rather than names keep the table narrow enough for one page.
  o <- unique(a[, c("country", "type", "brand", "outlet")])
  o <- o[order(o$country, o$type, o$brand), ]
  data.frame(Country = ifelse(duplicated(o$country), "", as.character(o$country)),
             Outlet = o$brand, key = o$outlet)
}

sum_by_outlet <- function(x, a, keys) tapply(x, a$outlet, sum, na.rm = TRUE)[keys]
pct <- function(x, total) sprintf("%.1f", 100 * x / total)   # one decimal, as text

# Column names "Issue.Column" become a spanning issue header via separate.header = TRUE
societal_composition <- function(a) {
  out <- outlet_frame(a)
  for (iss in names(ISSUE_LABEL)) {
    s      <- a[a$topic == iss, ]
    counts <- sapply(names(SOC_CATS), function(v) sum_by_outlet(s[[v]], s, out$key))
    total  <- rowSums(counts)
    for (v in names(SOC_CATS))
      out[[paste(ISSUE_LABEL[[iss]], SOC_CATS[[v]], sep = ".")]] <- pct(counts[, v], total)
  }
  out$key <- NULL
  out
}

political_composition <- function(a) {
  out <- outlet_frame(a)
  for (iss in names(ISSUE_LABEL)) {
    s     <- a[a$topic == iss, ]
    gov   <- sum_by_outlet(s$nr_actors * (s$government == 1), s, out$key)
    total <- sum_by_outlet(s$nr_actors, s, out$key)
    out[[ISSUE_LABEL[[iss]]]] <- pct(gov, total)
  }
  out$key <- NULL
  out
}


benchmark_table <- function() {
  # Table 3 runs on the country-week scores (one row per country and week), not the
  # outlet-week-issue grain the models use, so M and SD describe weekly variation.
  soc  <- read.csv(file.path("../model_dfs", "soc_div_country_week.csv"))
  pol  <- read.csv(file.path("../model_dfs", "pol_div_country_week.csv"))
  refl <- read.csv(file.path("../model_dfs", "pol_reflective_gov_share.csv"))
  refl <- setNames(refl$reflective_gov_share, refl$country)

  cc <- names(COUNTRY_NAME)
  f2 <- function(x) sprintf("%.2f", x)
  by_country <- function(d, col, fn) sapply(cc, function(k) fn(d[[col]][d$country == k]))

  # "Spanner.Column" names become the two spanning headers via separate.header = TRUE.
  # check.names = FALSE keeps the spaces and the delta sign, and lets M and SD repeat
  # once per block: R would otherwise rename the second pair M.1 and SD.1.
  data.frame(
    Country = unname(COUNTRY_NAME),
    "Societal actor diversity.M"                  = f2(by_country(soc, "simpson_Dz", mean)),
    "Societal actor diversity.SD"                 = f2(by_country(soc, "simpson_Dz", sd)),
    "Societal actor diversity.\u0394 open"        = f2(1 - by_country(soc, "simpson_Dz", mean)),
    "Political actor diversity.M"                 = f2(by_country(pol, "simpson_Dz", mean)),
    "Political actor diversity.SD"                = f2(by_country(pol, "simpson_Dz", sd)),
    "Political actor diversity.\u0394 open"       = f2(1 - by_country(pol, "simpson_Dz", mean)),
    "Political actor diversity.\u0394 reflective" =
      sprintf("%+.2f", 100 * (by_country(pol, "p_gov", mean) - refl[cc])),
    check.names = FALSE, row.names = NULL)
}


### Assumption checks

In [ ]:
check_assumptions <- function(m, d) {
  # Normality at each level, residual spread, and the random slope for issue by outlet.
  # print() is needed because only a function's last value is shown automatically.
  print(check_normality(m))                         # residuals
  print(check_normality(m, effects = "random"))     # each set of random intercepts
  print(datawizard::skewness(resid(m)))
  print(datawizard::kurtosis(resid(m)))
  print(check_heteroscedasticity(m))                # spread vs fitted values
  print(round(tapply(resid(m), d$country, sd), 3))  # spread by country

  m_slope <- update(m, . ~ . + (0 + dummy(iss, "rising prices") | outlet))
  print(anova(m, m_slope, refit = FALSE))

  par(bg = "white")                                 # visible on a dark theme
  qqnorm(resid(m)); qqline(resid(m), col = "red")
}

## 2. Descriptives

### 2.1 Diversity against the open and reflective benchmarks (Table 3)


In [ ]:
save_apa(benchmark_table(),
         "Table3_benchmarks", "3",
         "Societal and Political Actor Diversity Against the Open and Reflective Benchmarks",
         note = paste("Dz = standardized Simpson\u2019s diversity index, 1 at equal representation",
                      "of all categories. M and SD are over 53 country-weeks, 55 for Turkey.",
                      "\u0394 open = distance from the open benchmark (1 - M), \u0394 reflective =",
                      "the difference between % in coverage vs. % seats in the parliament in",
                      "percentage points, positive values indicate over-representation."),
         separate.header = TRUE)


### 2.2 Actor composition by outlet and issue (Tables C2 and C3)


In [ ]:
save_apa(societal_composition(read_articles("df_actors_socialdiv_analysis_clean.csv")),
         "TableC2_composition_societal", "C2",
         "Distribution of Societal Actor Categories by Outlet and Issue",
         note = paste("Percentages of all societal actors in an outlet's coverage of each issue,",
                      "summing to 100 across the four categories. Politics = government and",
                      "politics; Experts = professionals and experts. BE = Belgium; IR = Ireland;",
                      "NL = the Netherlands; TR = Turkey; UK = United Kingdom."),
         landscape = TRUE, separate.header = TRUE)

In [ ]:
save_apa(political_composition(read_articles("df_actors_politicaldiv_analysis_clean.csv")),
         "TableC3_composition_political", "C3",
         "Share of Governing-Party Actors by Outlet and Issue",
         note = paste("Percentages of political actors in an outlet's coverage of each issue",
                      "who are affiliated with a governing party; the remainder are affiliated",
                      "with opposition parties. BE = Belgium; IR = Ireland; NL = the Netherlands;",
                      "TR = Turkey; UK = United Kingdom."),
         landscape = TRUE)

### 2.3 Supplementary: composition by campaign period (not reported in the manuscript)


In [ ]:
societal_composition <- function(a, by = "topic", labels = ISSUE_LABEL) {
  # Share of each societal actor category per outlet, one column block per level of `by`
  # (issue by default; by = "campaign_period" with PERIOD_LABEL for the campaign table)
  out <- outlet_frame(a)
  for (lv in names(labels)) {
    s      <- a[!is.na(a[[by]]) & as.character(a[[by]]) == lv, ]
    counts <- sapply(names(SOC_CATS), function(v) sum_by_outlet(s[[v]], s, out$key))
    total  <- rowSums(counts)
    for (v in names(SOC_CATS))
      out[[paste(labels[[lv]], SOC_CATS[[v]], sep = ".")]] <- pct(counts[, v], total)
  }
  out$key <- NULL
  out
}

save_apa(societal_composition(read_articles("df_actors_socialdiv_analysis_clean.csv"),
                              by = "campaign_period", labels = PERIOD_LABEL),
         "TableS1_composition_societal_campaign", "S1",
         "Distribution of Societal Actor Categories by Outlet and Campaign Period",
         note = paste("Supplementary table, not reported in the manuscript.",
                      "Percentages of all societal actors in an outlet's coverage during routine",
                      "and campaign periods, summing to 100 across the four categories.",
                      "Politics = government and politics; Experts = professionals and experts.",
                      "BE = Belgium; IR = Ireland; NL = the Netherlands; TR = Turkey;",
                      "UK = United Kingdom."),
         landscape = TRUE, separate.header = TRUE)


In [ ]:
political_composition <- function(a, by = "topic", labels = ISSUE_LABEL) {
  # Governing-party share per outlet, one column per level of `by`
  # (issue by default; by = "campaign_period" with PERIOD_LABEL for the campaign table)
  out <- outlet_frame(a)
  for (lv in names(labels)) {
    s     <- a[!is.na(a[[by]]) & as.character(a[[by]]) == lv, ]
    gov   <- sum_by_outlet(s$nr_actors * (s$government == 1), s, out$key)
    total <- sum_by_outlet(s$nr_actors, s, out$key)
    out[[labels[[lv]]]] <- pct(gov, total)
  }
  out$key <- NULL
  out
}

save_apa(political_composition(read_articles("df_actors_politicaldiv_analysis_clean.csv"),
                               by = "campaign_period", labels = PERIOD_LABEL),
         "TableS2_composition_political_campaign", "S2",
         "Share of Governing-Party Actors by Outlet and Campaign Period",
         note = paste("Supplementary table, not reported in the manuscript.",
                      "Percentages of political actors in an outlet's coverage during routine",
                      "and campaign periods who are affiliated with a governing party; the",
                      "remainder are affiliated with opposition parties. BE = Belgium;",
                      "IR = Ireland; NL = the Netherlands; TR = Turkey; UK = United Kingdom."),
         landscape = TRUE)


## 3. Societal actor diversity

### 3.1 Data

In [ ]:
d_soc <- prep_data("soc_div_outlet_week_topic.csv")
summary(d_soc[, c("simpson_Dz", "country", "camp", "own", "iss")])

### 3.2 Model

In [ ]:
m_soc <- lmer(MODEL_FORMULA, data = d_soc, REML = TRUE)
print(VarCorr(m_soc), comp = "Variance")   # random effects; fixed effects are in the table below

### 3.3 Variance explained

In [ ]:
r2_nakagawa(m_soc)            # marginal R2 = fixed effects; conditional R2 = fixed + random

# Share of the TOTAL variance from each source (the four add up to 100%). Not the ICC,
# which divides by the random + residual variance only and so gives larger numbers.
v <- insight::get_variance(m_soc)
round(100 * c(fixed = v$var.fixed, v$var.intercept, residual = v$var.residual) /
      (v$var.fixed + v$var.random + v$var.residual), 1)

### 3.4 Assumptions

In [ ]:
check_assumptions(m_soc, d_soc)

### 3.5 Model and country comparisons (Tables 4 and D1)

In [ ]:
save_apa(model_table(m_soc), "Table4_model_societal", 4,
         "Mixed-Effects Model of Societal Actor Diversity")

In [ ]:
save_apa(tidy_emm(emmeans(m_soc, ~ country)), "Country_means_societal", "X",
         "Estimated Marginal Means of Societal Actor Diversity by Country")

In [ ]:
save_apa(pairs_table(m_soc), "TableD1_country_contrasts_societal", "D1",
         "Pairwise Comparisons of Societal Actor Diversity, Derived From the Mixed Model",
         note = "Holm-adjusted p values control the familywise error rate across the ten comparisons.")

### 3.6 Within-country effects (Table D2)

In [ ]:
save_apa(simple_effects_table(m_soc), "TableD2_within_country_societal", "D2",
         "Effects of Campaign Period, Ownership and Issue on Societal Actor Diversity, Within Each Country",
         note = "Estimates derived from the model in Table 4. Unadjusted p values.")

## 4. Political actor diversity

### 4.1 Data

In [ ]:
d_pol <- prep_data("pol_div_outlet_week_topic.csv")
summary(d_pol[, c("simpson_Dz", "country", "camp", "own", "iss")])

### 4.2 Model

In [ ]:
m_pol <- lmer(MODEL_FORMULA, data = d_pol, REML = TRUE)
print(VarCorr(m_pol), comp = "Variance")   # random effects; fixed effects are in the table below

### 4.3 Variance explained

In [ ]:
r2_nakagawa(m_pol)            # marginal R2 = fixed effects; conditional R2 = fixed + random

# Share of the TOTAL variance from each source (the four add up to 100%). Not the ICC,
# which divides by the random + residual variance only and so gives larger numbers.
v <- insight::get_variance(m_pol)
round(100 * c(fixed = v$var.fixed, v$var.intercept, residual = v$var.residual) /
      (v$var.fixed + v$var.random + v$var.residual), 1)

### 4.4 Assumptions

In [ ]:
check_assumptions(m_pol, d_pol)

### 4.5 Model and country comparisons (Tables 5 and D3)

In [ ]:
save_apa(model_table(m_pol), "Table5_model_political", 5,
         "Mixed-Effects Model of Political Actor Diversity")

In [ ]:
save_apa(tidy_emm(emmeans(m_pol, ~ country)), "Country_means_political", "X",
         "Estimated Marginal Means of Political Actor Diversity by Country")

In [ ]:
save_apa(pairs_table(m_pol), "TableD3_country_contrasts_political", "D3",
         "Pairwise Comparisons of Political Actor Diversity, Derived From the Mixed Model",
         note = "Holm-adjusted p values control the familywise error rate across the ten comparisons.")

### 4.6 Within-country effects (Table D4)

In [ ]:
save_apa(simple_effects_table(m_pol), "TableD4_within_country_political", "D4",
         "Effects of Campaign Period, Ownership and Issue on Political Actor Diversity, Within Each Country",
         note = "Estimates derived from the model in Table 5. Unadjusted p values.")